In [1]:
# Import packages
import os
import datetime
from datetime import datetime
import sys

sys.path.append("/Users/apple/repos/IKIEnv/lib/python3.7/site-packages/")

import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 75)
pd.set_option('display.max_rows', 300)
pd.options.mode.chained_assignment = None

# UNCTAD Trade-and-Transport Dataset - Portfolios Generation Script

<h4>Conclusions from 'Explorer' Script:</h4>

<li>The resulting dataset represents USD10.6tn. The online Trade-and-Transport dashboard indicates this figure is USD16.5tn, so this discrepancy needs to be looked into further. There's some preliminary evidence that the losses occuring from linking with country information represents around $0.6tn only.</li>

<h4>Completed in this Script:</h4>

<ol type=1>
    <li>The 'CLEAN' Trade-and-Transport database is used to derive international trade portfolios for visualisation on the Maritime Tracker platform.</li>
</ol>

In [2]:
# Import packages
import os
import datetime
from datetime import datetime
import sys

sys.path.append("/Users/apple/repos/IKIEnv/lib/python3.7/site-packages/")

import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 75)
pd.set_option('display.max_rows', 300)
pd.options.mode.chained_assignment = None

# 1. Reading in 'Clean' Trade-and-Transport Database for 2018

In [10]:
trade_trans_raw_r = {"Product":"HS2"}
trade_trans_d = {"Origin":"str", "Origin Label":"str", "Destination":"str", "Destination Label":"str", "Product":"str", "TransportMode":"str"}
trade_trans_raw = pd.read_csv("/Users/apple/repos/datasets/unctad/trade_and_transport/data/US_TransportCosts_20182018_CLEAN.csv", dtype=trade_trans_d).rename(columns=trade_trans_raw_r)

HS2s_to_eora = pd.read_csv("/Users/apple/repos/datasets/commodity_conversions/HS2s_to_eora.csv", dtype={"HS2":"str"})
trade_trans = pd.merge(trade_trans_raw, HS2s_to_eora, left_on="HS2", right_on="HS2", how="left")
display(trade_trans.iloc[:1])

HS2s_by_mode = trade_trans[(trade_trans.HS2.str.len() == 2)]
HS2s_by_mode["tonne"] = HS2s_by_mode["Kilograms"] / 1e3
HS2s_by_mode = HS2s_by_mode.drop(columns=["Kilograms"])

print("Value of 'HS2s_by_mode.csv': ${0}tn".format(np.round(HS2s_by_mode["FOB value (US$)"].sum() / 1e12, 1)))
print("Weight of 'HS2s_by_mode.csv': {0}bn tonnes".format(np.round(HS2s_by_mode["tonne"].sum() / 1e9, 1)))
display(HS2s_by_mode.iloc[:1])

HS2s_by_sea = HS2s_by_mode[(HS2s_by_mode["TransportMode Label"] == "Sea")]
print("Value of 'HS2s_by_sea.csv': ${0}tn".format(np.round(HS2s_by_sea["FOB value (US$)"].sum() / 1e12, 1)))
print("Weight of 'HS2s_by_sea.csv': {0}bn tonnes".format(np.round(HS2s_by_sea["tonne"].sum() / 1e9, 1)))
display(HS2s_by_sea.iloc[:1])

HS2s_by_sea_c_r = {
    "Origin":"exp_code_str3", "Origin Label":"exp_name", 
    "Destination":"imp_code_str3", "Destination Label":"imp_name", 
    "FOB value (US$)":"USD"}

HS2s_by_sea_c = HS2s_by_sea.rename(columns=HS2s_by_sea_c_r)
display(HS2s_by_sea_c.iloc[:1])

,Origin,Origin Label,Destination,Destination Label,HS2,TransportMode,TransportMode Label,FOB value (US$),Kilograms,description,eora_group
0,004,Afghanistan,004,Afghanistan,TOTAL,80,Multimodal adjustment,NaN,NaN,NaN,NaN


Value of 'HS2s_by_mode.csv': $16.5tn
Weight of 'HS2s_by_mode.csv': 12.7bn tonnes


,Origin,Origin Label,Destination,Destination Label,HS2,TransportMode,TransportMode Label,FOB value (US$),description,eora_group,tonne
1,004,Afghanistan,004,Afghanistan,01,80,Multimodal adjustment,NaN,"Horses, live pure-bred breeding",agriculture,NaN


Value of 'HS2s_by_sea.csv': $6.5tn
Weight of 'HS2s_by_sea.csv': 6.3bn tonnes


,Origin,Origin Label,Destination,Destination Label,HS2,TransportMode,TransportMode Label,FOB value (US$),description,eora_group,tonne
2714,004,Afghanistan,012,Algeria,08,21,Sea,0.0,"Coconuts, fresh or dried",agriculture,0.0


,exp_code_str3,exp_name,imp_code_str3,imp_name,HS2,TransportMode,TransportMode Label,USD,description,eora_group,tonne
2714,004,Afghanistan,012,Algeria,08,21,Sea,0.0,"Coconuts, fresh or dried",agriculture,0.0


## 1.1 Align TandT Country-codes with 'National Stats'

In [11]:
country_iso_codes_cols = ["name", "alpha-2", "alpha-3", "country-code"]
country_iso_codes_dtype = {"alpha-2":str, "alpha-3":str, "country-code":str}
country_iso_codes_renames = {"name":"iso_country", "alpha-2":"iso_2", "alpha-3":"iso_3", "country-code":"iso_code"}

# Read-in ISO Country Labels dataset and use as Standard
country_iso_codes = pd.read_csv("/Users/apple/repos/datasets/national_stats/country_iso_codes.csv", usecols=country_iso_codes_cols, dtype=country_iso_codes_dtype).rename(columns=country_iso_codes_renames)
country_iso_codes.loc[country_iso_codes.iso_country == "Namibia", "iso_2"] = "NA"
display(country_iso_codes.iloc[:2])

Os = HS2s_by_sea_c[["exp_code_str3", "exp_name"]].drop_duplicates().rename(columns={"exp_code_str3":"iso_code", "exp_name":"iso_name"})
Ds = HS2s_by_sea_c[["imp_code_str3", "imp_name"]].drop_duplicates().rename(columns={"imp_code_str3":"iso_code", "imp_name":"iso_name"})
ODs = pd.concat([Os, Ds], axis=0).drop_duplicates().reset_index(drop=True)
ODs_not = ODs[(~ODs.iso_code.isin(country_iso_codes.iso_code.values))]
display(ODs_not)

,iso_country,iso_2,iso_3,iso_code
0,Afghanistan,AF,AFG,004
1,Åland Islands,AX,ALA,248


,iso_code,name
74,251,France
150,579,Norway
193,757,"Switzerland, Liechtenstein"
211,926,United Kingdom
212,842,United States of America


In [23]:
# Provide a clean iso_code field where current value is wrong based on the above.
HS2s_by_sea_c.loc[HS2s_by_sea_c.exp_code_str3 == "251", "exp_code_str3"] = "250" # France
HS2s_by_sea_c.loc[HS2s_by_sea_c.imp_code_str3 == "251", "imp_code_str3"] = "250"
HS2s_by_sea_c.loc[HS2s_by_sea_c.exp_code_str3 == "579", "exp_code_str3"] = "578" # Norway
HS2s_by_sea_c.loc[HS2s_by_sea_c.imp_code_str3 == "579", "imp_code_str3"] = "578"
HS2s_by_sea_c.loc[HS2s_by_sea_c.exp_code_str3 == "757", "exp_code_str3"] = "756" # Switzerland
HS2s_by_sea_c.loc[HS2s_by_sea_c.imp_code_str3 == "757", "imp_code_str3"] = "756"
HS2s_by_sea_c.loc[HS2s_by_sea_c.exp_code_str3 == "842", "exp_code_str3"] = "840" # USA
HS2s_by_sea_c.loc[HS2s_by_sea_c.imp_code_str3 == "842", "imp_code_str3"] = "840"
HS2s_by_sea_c.loc[HS2s_by_sea_c.exp_code_str3 == "926", "exp_code_str3"] = "826" # UK
HS2s_by_sea_c.loc[HS2s_by_sea_c.imp_code_str3 == "926", "imp_code_str3"] = "826"

In [24]:
Os = HS2s_by_sea_c[["exp_code_str3", "exp_name"]].drop_duplicates().rename(columns={"exp_code_str3":"iso_code", "exp_name":"iso_name"})
Ds = HS2s_by_sea_c[["imp_code_str3", "imp_name"]].drop_duplicates().rename(columns={"imp_code_str3":"iso_code", "imp_name":"iso_name"})
ODs = pd.concat([Os, Ds], axis=0).drop_duplicates().reset_index(drop=True)
ODs_not = ODs[(~ODs.iso_code.isin(country_iso_codes.iso_code.values))]
display(ODs_not)

,iso_code,iso_name


# 2. 

In [12]:
def exp_merch_trade_stats_gen(X):

    ## EXPORTER SIDE
    # Top 15 Most Valuable Trade Flows by Value and Weight
    X_tr_usd = X.groupby(["exp_code_str3", "exp_name", "imp_code_str3", "imp_name", "HS2", "description"]).agg({"USD":"sum", "tonne":"sum"}).sort_values(by="USD", ascending=False).reset_index()
    X_tr_usd["clean_desc"] = X_tr_usd.imp_name + " - " + X_tr_usd.description
    X_tr_t = X.groupby(["exp_code_str3", "exp_name", "imp_code_str3", "imp_name", "HS2", "description"]).agg({"USD":"sum", "tonne":"sum"}).sort_values(by="tonne", ascending=False).reset_index()
    X_tr_t["clean_desc"] = X_tr_t.imp_name + " - " + X_tr_t.description
    
    # Top 15 Most Valuable Commodity Flows by Value and Weight
    X_co_usd = X.groupby(["exp_code_str3", "exp_name", "HS2", "description"]).agg({"USD":"sum", "tonne":"sum"}).sort_values(by="USD", ascending=False).reset_index()
    X_co_t = X.groupby(["exp_code_str3", "exp_name", "HS2", "description"]).agg({"USD":"sum", "tonne":"sum"}).sort_values(by="tonne", ascending=False).reset_index()
    
    # Top 15 Most Valuable Trade Partners
    X_pa_usd = X.groupby(["exp_code_str3", "exp_name", "imp_code_str3", "imp_name"]).agg({"USD":"sum", "tonne":"sum"}).sort_values(by="USD", ascending=False).reset_index()
    X_pa_t = X.groupby(["exp_code_str3", "exp_name", "imp_code_str3", "imp_name"]).agg({"USD":"sum", "tonne":"sum"}).sort_values(by="tonne", ascending=False).reset_index()
    
    return X_tr_usd, X_tr_t, X_co_usd, X_co_t, X_pa_usd, X_pa_t

def imp_merch_trade_stats_gen(I):

    ## IMPORTER SIDE
    # Top 15 Most Valuable Trade Flows by Value and Weight
    I_tr_usd = I.groupby(["exp_code_str3", "exp_name", "imp_code_str3", "imp_name", "HS2", "description"]).agg({"USD":"sum", "tonne":"sum"}).sort_values(by="USD", ascending=False).reset_index()
    I_tr_usd["clean_desc"] = I_tr_usd.exp_name + " - " + I_tr_usd.description
    I_tr_t = I.groupby(["exp_code_str3", "exp_name", "imp_code_str3", "imp_name", "HS2", "description"]).agg({"USD":"sum", "tonne":"sum"}).sort_values(by="tonne", ascending=False).reset_index()
    I_tr_t["clean_desc"] = I_tr_t.exp_name + " - " + I_tr_t.description
    
    # Top 15 Most Valuable Commodity Flows by Value and Weight
    I_co_usd = I.groupby(["imp_code_str3", "imp_name", "HS2", "description"]).agg({"USD":"sum", "tonne":"sum"}).sort_values(by="USD", ascending=False).reset_index()
    I_co_t = I.groupby(["imp_code_str3", "imp_name", "HS2", "description"]).agg({"USD":"sum", "tonne":"sum"}).sort_values(by="tonne", ascending=False).reset_index()

    # Top 15 Most Valuable Trade Partners
    I_pa_usd = I.groupby(["exp_code_str3", "exp_name", "imp_code_str3", "imp_name"]).agg({"USD":"sum", "tonne":"sum"}).sort_values(by="USD", ascending=False).reset_index()
    I_pa_t = I.groupby(["exp_code_str3", "exp_name", "imp_code_str3", "imp_name"]).agg({"USD":"sum", "tonne":"sum"}).sort_values(by="tonne", ascending=False).reset_index()

    return I_tr_usd, I_tr_t, I_co_usd, I_co_t, I_pa_usd, I_pa_t

In [27]:
counter = 0
output_dir = "/Users/apple/repos/datasets/unctad/trade_and_transport/portfolios/"

for iso_idx, iso in ODs.iterrows():

    # Create output folder
    iso_id = iso.iso_code + " - " + iso.iso_name
    print("Processing Export Trade Flows for {0} (country {1} of {2}).".format(iso_id, iso_idx + 1, len(ODs))) if counter in range(0, 300, 20) else False
    counter += 1

    # Create directory if necessary
    if not os.path.exists(output_dir+"{0}/".format(iso.iso_code)):
        os.makedirs(output_dir+"{0}/".format(iso.iso_code)) 
    
    ## EXPORTER SIDE
    X = HS2s_by_sea_c[HS2s_by_sea_c.exp_code_str3 == iso.iso_code]
    X_tr_usd, X_tr_t, X_co_usd, X_co_t, X_pa_usd, X_pa_t = exp_merch_trade_stats_gen(X)
    
    # Top 15 Most Valuable Trade Flows by Value and Weight
    X_tr_usd.to_csv(output_dir+"{0}/{1}".format(iso.iso_code, "X_tr_usd.csv"))
    X_tr_t.to_csv(output_dir+"{0}/{1}".format(iso.iso_code, "X_tr_t.csv"))
        
    # Top 15 Most Valuable Commodity Flows by Value and Weight
    X_co_usd.to_csv(output_dir+"{0}/{1}".format(iso.iso_code, "X_co_usd.csv"))
    X_co_t.to_csv(output_dir+"{0}/{1}".format(iso.iso_code, "X_co_t.csv"))
    
    # Top 15 Most Valuable Trade Partners
    X_pa_usd.to_csv(output_dir+"{0}/{1}".format(iso.iso_code, "X_pa_usd.csv"))
    X_pa_t.to_csv(output_dir+"{0}/{1}".format(iso.iso_code, "X_pa_t.csv"))

    
    ## IMPORTER SIDE
    I = HS2s_by_sea_c[HS2s_by_sea_c.imp_code_str3 == iso.iso_code]
    I_tr_usd, I_tr_t, I_co_usd, I_co_t, I_pa_usd, I_pa_t = imp_merch_trade_stats_gen(I)
    
    # Top 15 Most Valuable Trade Flows by Value and Weight
    I_tr_usd.to_csv(output_dir+"{0}/{1}".format(iso.iso_code, "I_tr_usd.csv"))
    I_tr_t.to_csv(output_dir+"{0}/{1}".format(iso.iso_code, "I_tr_t.csv"))
        
    # Top 15 Most Valuable Commodity Flows by Value and Weight
    I_co_usd.to_csv(output_dir+"{0}/{1}".format(iso.iso_code, "I_co_usd.csv"))
    I_co_t.to_csv(output_dir+"{0}/{1}".format(iso.iso_code, "I_co_t.csv"))
    
    # Top 15 Most Valuable Trade Partners
    I_pa_usd.to_csv(output_dir+"{0}/{1}".format(iso.iso_code, "I_pa_usd.csv"))
    I_pa_t.to_csv(output_dir+"{0}/{1}".format(iso.iso_code, "I_pa_t.csv"))

Processing Export Trade Flows for 004 - Afghanistan (country 1 of 222).
Processing Export Trade Flows for 084 - Belize (country 21 of 222).
Processing Export Trade Flows for 148 - Chad (country 41 of 222).
Processing Export Trade Flows for 212 - Dominica (country 61 of 222).
Processing Export Trade Flows for 288 - Ghana (country 81 of 222).
Processing Export Trade Flows for 380 - Italy (country 101 of 222).
Processing Export Trade Flows for 454 - Malawi (country 121 of 222).
Processing Export Trade Flows for 528 - Netherlands (Kingdom of the) (country 141 of 222).
Processing Export Trade Flows for 620 - Portugal (country 161 of 222).
Processing Export Trade Flows for 534 - Sint Maarten (Dutch part) (country 181 of 222).
Processing Export Trade Flows for 772 - Tokelau (country 201 of 222).
Processing Export Trade Flows for 894 - Zambia (country 221 of 222).
